# 🔴 Hard: Conv2D Backward (NumPy)

Given the upstream gradient `dout`, compute `dx`, `dw` and `db` for a 2-D convolution — pure NumPy.

### Core Idea

Convolution is linear in **both** `x` and `w`, so its backward pass is two contractions of the same
sliding window — the exact mirror of the forward loop.

For one output position $(i, j)$, write $d = \text{dout}[:, :, i, j]$ with shape $(N, F)$ and
$\text{patch} = x_\text{pad}[:, :, is{:}is{+}K_H,\ js{:}js{+}K_W]$ with shape $(N, C, K_H, K_W)$:

$$db_f = \sum_{n, i, j} \text{dout}[n, f, i, j] \qquad\text{— the bias is broadcast, so it sums back}$$

$$dw \mathrel{+}= \sum_n d[n, :] \otimes \text{patch}[n] \qquad dx_\text{pad}[\text{window}] \mathrel{+}= \sum_f d[:, f]\, w[f]$$

The single idea that makes it click: **weight sharing forward ⇒ gradient accumulation backward.**
A weight used at 64 positions collects gradient from all 64 — that is why `dw` is a *sum over the
whole feature map*, and why `+=` (never `=`) is mandatory when writing into `dx`. With `stride <
kernel_size` the windows overlap, so an interior pixel feeds several outputs and must collect all of
their gradients.

Two loose ends that are pure bookkeeping but where everyone loses an hour:

- Accumulate into the **padded** `dx`, then crop the border off at the very end — `dx` must come back
  with the shape of `x`, not of `x_pad`.
- `dw` sums over the batch too. Its shape is `(F, C, KH, KW)` no matter how large `N` is.

Fun fact for later: $dx$ is itself a convolution of `dout` with the *flipped* kernel — a
"transposed convolution", the same op that upsamples in decoders and GANs.

### Signature
```python
def conv2d_backward(dout, x, w, stride=1, padding=0):
    # dout: (N, F, H_out, W_out) upstream gradient
    # x:    (N, C, H, W)   w: (F, C, KH, KW)
    # returns: (dx, dw, db) with shapes of x, w, (F,)
    ...
```

### Rules
- Pure **NumPy** — no PyTorch, no autograd
- Accumulate with `+=`; overlapping windows must add up
- Crop the padding off `dx` before returning

### Example
```
dx, dw, db = conv2d_backward(dout, x, w, stride=1, padding=1)
# db == dout.sum(axis=(0, 2, 3))
# matches tx.grad / tw.grad from torch autograd
```

In [ ]:
import numpy as np

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def conv2d_backward(dout, x, w, stride=1, padding=0):
    # dout: (N, F, H_out, W_out);  x: (N, C, H, W);  w: (F, C, KH, KW)
    # returns (dx, dw, db)
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
np.random.seed(0)
x = np.random.randn(2, 3, 8, 8)
w = np.random.randn(4, 3, 3, 3)
dout = np.random.randn(2, 4, 6, 6)

dx, dw, db = conv2d_backward(dout, x, w, stride=1, padding=0)
print("dx:", dx.shape, " dw:", dw.shape, " db:", db.shape)
print("db == dout.sum:", np.allclose(db, dout.sum(axis=(0, 2, 3))))

# Overlapping windows: an interior pixel is covered by 4 windows of a 2x2 kernel at stride 1
xs = np.zeros((1, 1, 4, 4))
ws = np.ones((1, 1, 2, 2))
dxs, _, _ = conv2d_backward(np.ones((1, 1, 3, 3)), xs, ws, stride=1, padding=0)
print("corner pixel  :", dxs[0, 0, 0, 0], "(expect 1.0)")
print("interior pixel:", dxs[0, 0, 1, 1], "(expect 4.0)")

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("numpy_conv2d_backward")